In [1]:
import torch
import torch.nn.functional as F
from torch.cuda import device
from torch_geometric.nn import GATConv # ИМПОРТИРУЕМ GAT!

class GAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes, heads=8):
        super(GAT, self).__init__()

        # --- 1. Скрытый слой ---
        # Выход одного узла: 8 признаков. Но у нас 8 голов (heads=8)!
        # GATConv склеит выходы всех голов, поэтому реальный размер на выходе
        # будет 8 * 8 = 64 признака.
        self.conv1 = GATConv(in_channels=num_node_features,
                             out_channels=8,
                             heads=heads,
                             dropout=0.6) # В GAT встроен свой dropout для внимания

        # --- 2. Выходной слой ---
        # Вход: 64 признака (8 каналов * 8 голов от прошлого слоя).
        # Выход: 7 вероятностей тем.
        # Для последнего слоя мы ставим concat=False (усредняем головы, а не склеиваем).
        self.conv2 = GATConv(in_channels=8 * heads,
                             out_channels=num_classes,
                             heads=1, # На выходе 1 голова (или усреднение нескольких)
                             concat=False,
                             dropout=0.6)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Проходим первый слой
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x) # ELU работает для GAT лучше, чем ReLU (рекомендация автора статьи)

        # Проходим второй слой
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

In [2]:
import torch
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='../data/Cora', name='Cora')

data = dataset[0]

print(f"Количество классов (тем статей): {dataset.num_classes}")
print(f"Количество признаков (длина словаря): {dataset.num_node_features}")
print("---")
print("Сам граф:")
data

Количество классов (тем статей): 7
Количество признаков (длина словаря): 1433
---
Сам граф:


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
device

device(type='cuda')

In [8]:
model = GAT(num_node_features=dataset.num_node_features, num_classes=dataset.num_classes, heads=8).to(device)

In [9]:
# 1. Настройки оптимизатора (Adam - наш лучший друг)
# weight_decay - это L2-регуляризация, спасает от переобучения
# Правильный оптимизатор для GAT
optimizer = torch.optim.Adam([
    dict(params=model.conv1.parameters(), weight_decay=5e-4),
    dict(params=model.conv2.parameters(), weight_decay=0)
], lr=0.005)
# Переводим всё на видеокарту (если есть)
data = data.to(device)
model.train()

epochs = 200 # Для GNN часто нужно чуть больше эпох, они быстрые

for epoch in range(epochs):
    optimizer.zero_grad()

    # 2. ПРЯМОЙ ПРОХОД: Закидываем ВЕСЬ граф целиком
    out = model(data)

    # 3. СЧИТАЕМ ОШИБКУ (LOSS) ТОЛЬКО ДЛЯ ТРЕНИРОВОЧНЫХ УЗЛОВ!
    # Наш выход out имеет размер [2708, 7].
    # out[data.train_mask] вырежет оттуда только те 140 строк, на которых мы учимся.
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])

    # 4. ОБРАТНЫЙ ПРОХОД (Градиенты текут по связям!)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f'Эпоха: {epoch:03d}, Loss: {loss:.4f}')

print("Обучение завершено!")

Эпоха: 000, Loss: 2.0314
Эпоха: 020, Loss: 0.8555
Эпоха: 040, Loss: 0.5553
Эпоха: 060, Loss: 0.3699
Эпоха: 080, Loss: 0.4142
Эпоха: 100, Loss: 0.4726
Эпоха: 120, Loss: 0.4388
Эпоха: 140, Loss: 0.3786
Эпоха: 160, Loss: 0.3892
Эпоха: 180, Loss: 0.4174
Обучение завершено!


In [10]:
# Переводим модель в режим оценки (отключаем Dropout)
model.eval()

# Делаем финальное предсказание (градиенты больше не нужны)
with torch.no_grad():
    out = model(data)

    # argmax(dim=1) берет индекс максимальной вероятности (наш предсказанный класс)
    pred = out.argmax(dim=1)

    # Сравниваем предсказания с реальными ответами, НО ТОЛЬКО для тестовой маски
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()

    # Считаем процент правильных ответов
    test_acc = int(correct) / int(data.test_mask.sum())

    print(f'Точность на тестовой выборке: {test_acc:.4f} ({test_acc * 100:.2f}%)')

Точность на тестовой выборке: 0.7830 (78.30%)


In [11]:
import itertools
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv

# 1. Модернизированный класс, который принимает dropout в качестве аргумента
class ParametricGAT(torch.nn.Module):
    def __init__(self, num_features, num_classes, hidden_channels=8, heads=8, dropout_rate=0.6):
        super(ParametricGAT, self).__init__()
        self.dropout_rate = dropout_rate

        self.conv1 = GATConv(num_features, hidden_channels, heads=heads, dropout=dropout_rate)
        # На втором слое 1 голова (concat=False)
        self.conv2 = GATConv(hidden_channels * heads, num_classes, heads=1, concat=False, dropout=dropout_rate)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)

        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

# 2. Функция обучения и оценки для одной комбинации параметров
def train_and_evaluate(lr, weight_decay, dropout_rate, heads):
    # Фиксируем случайность, чтобы результаты не прыгали
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    model = ParametricGAT(dataset.num_node_features, dataset.num_classes,
                          hidden_channels=8, heads=heads, dropout_rate=dropout_rate).to(device)

    # Хитрость из статьи: L2-регуляризация только для первого слоя
    optimizer = torch.optim.Adam([
        dict(params=model.conv1.parameters(), weight_decay=weight_decay),
        dict(params=model.conv2.parameters(), weight_decay=0)
    ], lr=lr)

    model.train()
    for epoch in range(200): # 200 эпох вполне достаточно
        optimizer.zero_grad()
        out = model(data)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

    # Оценка
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)
        correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
        test_acc = int(correct) / int(data.test_mask.sum())

    return test_acc

# 3. Настраиваем сетку параметров (Grid)
lrs = [0.005, 0.01]               # Скорость обучения
weight_decays = [5e-4, 1e-3]      # Сила штрафа за большие веса
dropouts = [0.5, 0.6]             # Процент отключенных нейронов
heads_list = [8]                  # Количество голов (оставляем 8 по классике)

best_acc = 0
best_params = None

print("Запускаем Grid Search для GAT...\n")

# Перебираем все возможные комбинации с помощью itertools.product
for lr, wd, drop, heads in itertools.product(lrs, weight_decays, dropouts, heads_list):
    acc = train_and_evaluate(lr, wd, drop, heads)
    print(f"LR: {lr:5.3f} | L2: {wd:6.4f} | Drop: {drop} | Heads: {heads}  =>  Точность: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_params = (lr, wd, drop, heads)

print("-" * 50)
print(f"🏆 Лучшая точность GAT: {best_acc:.4f} ({(best_acc*100):.2f}%)")
print(f"Оптимальные параметры: LR={best_params[0]}, L2={best_params[1]}, Drop={best_params[2]}")

Запускаем Grid Search для GAT...

LR: 0.005 | L2: 0.0005 | Drop: 0.5 | Heads: 8  =>  Точность: 0.8040
LR: 0.005 | L2: 0.0005 | Drop: 0.6 | Heads: 8  =>  Точность: 0.8010
LR: 0.005 | L2: 0.0010 | Drop: 0.5 | Heads: 8  =>  Точность: 0.8070
LR: 0.005 | L2: 0.0010 | Drop: 0.6 | Heads: 8  =>  Точность: 0.8050
LR: 0.010 | L2: 0.0005 | Drop: 0.5 | Heads: 8  =>  Точность: 0.8020
LR: 0.010 | L2: 0.0005 | Drop: 0.6 | Heads: 8  =>  Точность: 0.7950
LR: 0.010 | L2: 0.0010 | Drop: 0.5 | Heads: 8  =>  Точность: 0.8030
LR: 0.010 | L2: 0.0010 | Drop: 0.6 | Heads: 8  =>  Точность: 0.8080
--------------------------------------------------
🏆 Лучшая точность GAT: 0.8080 (80.80%)
Оптимальные параметры: LR=0.01, L2=0.001, Drop=0.6
